In [ ]:
import numpy as np
import re
import io
import base64
from itertools import combinations

import networkx as nx
from networkx.generators.atlas import graph_atlas_g
import matplotlib.pyplot as plt
import pandas as pd
import altair as alt

In [ ]:
def load_grid(filepath):
    with open(filepath, 'r') as file:
        lines = file.readlines()
    return lines


def build_edges(rows, cols):
    edges = []
    for r in range(rows):
        for c in range(cols):
            n = r * cols + c
            if c < cols - 1:
                edges.append((n, n + 1))
            if r < rows - 1:
                edges.append((n, n + cols))
    edges = np.array(edges)
    return edges[:, 0], edges[:, 1]


def cut_edges(a, U, V):
    return int(np.sum(a[U] != a[V]))


def parse(line):
    return np.array([int(x) for x in line.strip().split(',')], dtype=int)


def district_nodes(a, d):
    return np.flatnonzero(a == d)


def induced_edges(U, V, nodes):
    keep = np.isin(U, nodes) & np.isin(V, nodes)
    return U[keep], V[keep]


def laplacian(nodes, U_sub, V_sub):
    index_of = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    L = np.zeros((n, n), dtype=int)
    for u, v in zip(U_sub, V_sub):
        i, j = index_of[u], index_of[v]
        L[i, i] += 1
        L[j, j] += 1
        L[i, j] -= 1
        L[j, i] -= 1
    return L


def spanning_trees(nodes, U_sub, V_sub):
    if len(nodes) <= 1:
        return 1
    L = laplacian(nodes, U_sub, V_sub)
    minor = L[1:, 1:]
    return round(np.linalg.det(minor))


def plan_spanning_trees(a, U, V):
    total = 1
    for d in np.unique(a):
        nodes = district_nodes(a, d)
        U_sub, V_sub = induced_edges(U, V, nodes)
        total *= spanning_trees(nodes, U_sub, V_sub)
    return total


def get_spanning_trees(assignments, rows, cols):
    U, V = build_edges(rows, cols)
    return [plan_spanning_trees(parse(line), U, V) for line in assignments]


def ground_truth_distribution(assignments, rows, cols):
    weights = np.array(get_spanning_trees(assignments, rows, cols), dtype=float)
    Z = weights.sum()
    return weights / Z


def district_adjacency(a, U, V):
    pairs = set()
    for u, v in zip(U, V):
        du, dv = a[u], a[v]
        if du != dv:
            pairs.add(tuple(sorted((du, dv))))
    return pairs


def build_plan_index(assignments, rows, cols):
    index = {}
    for i, line in enumerate(assignments):
        key = canonicalize(parse(line))
        index[key] = i
    return index


def canonicalize(a):
    mapping = {}
    next_label = 1
    canon = np.empty_like(a)
    for i, val in enumerate(a):
        if val not in mapping:
            mapping[val] = next_label
            next_label += 1
        canon[i] = mapping[val]
    return tuple(canon)


def enumerate_spanning_trees(nodes, U_sub, V_sub):
    edge_list = [(U_sub[i], V_sub[i], i) for i in range(len(U_sub))]

    def connected(node_list, edges):
        if len(node_list) <= 1:
            return True
        parent = {v: v for v in node_list}
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        remaining = len(node_list)
        for u, v, _ in edges:
            ru, rv = find(u), find(v)
            if ru != rv:
                parent[ru] = rv
                remaining -= 1
                if remaining == 1:
                    return True
        return False

    def recurse(node_list, edges):
        if len(node_list) == 1:
            return [[]]
        if not connected(node_list, edges):
            return []
        u, v, eid = edges[0]
        rest = edges[1:]

        trees_without = recurse(node_list, rest)

        merged_nodes = [x for x in node_list if x != v]
        merged_edges = []
        for (a, b, e2) in rest:
            a = u if a == v else a
            b = u if b == v else b
            if a == b:
                continue
            merged_edges.append((a, b, e2))
        trees_with = [
            t + [(U_sub[eid], V_sub[eid], eid)]
            for t in recurse(merged_nodes, merged_edges)
        ]
        return trees_without + trees_with

    return recurse(list(nodes), edge_list)


def balanced_edges(tree, target_size):
    adj = {}
    for (u, v, eid) in tree:
        adj.setdefault(u, []).append((v, eid))
        adj.setdefault(v, []).append((u, eid))

    results = []
    for (u, v, eid) in tree:
        visited = {u}
        stack = [u]
        while stack:
            cur = stack.pop()
            for nbr, e2 in adj[cur]:
                if e2 == eid:
                    continue
                if nbr not in visited:
                    visited.add(nbr)
                    stack.append(nbr)
        if len(visited) == target_size:
            results.append((eid, frozenset(visited)))
    return results


def apply_split(a, W, i, j, side_nodes):
    new_a = a.copy()
    other_side = set(W.tolist()) - side_nodes
    new_a[list(side_nodes)] = i
    new_a[list(other_side)] = j
    return new_a


def shape_split_weights(W, U_sub, V_sub, target_size):
    trees = enumerate_spanning_trees(W, U_sub, V_sub)
    total_trees = len(trees)
    raw_weights = {}
    for tree in trees:
        cuts = balanced_edges(tree, target_size)
        if not cuts:
            continue
        w = 1.0 / len(cuts)
        for _eid, side_nodes in cuts:
            raw_weights[side_nodes] = raw_weights.get(side_nodes, 0.0) + w
    return total_trees, raw_weights


def split_distribution(a, W, U_sub, V_sub, i, j, target_size, shape_cache=None):
    key = tuple(int(x) for x in W)
    cached = None if shape_cache is None else shape_cache.get(key)
    if cached is None:
        cached = shape_split_weights(W, U_sub, V_sub, target_size)
        if shape_cache is not None:
            shape_cache[key] = cached
    total_trees, raw_weights = cached
    if total_trees == 0:
        return {}

    split_prob = {}
    for side_nodes, w in raw_weights.items():
        new_a = apply_split(a, W, i, j, side_nodes)
        s = cut_edges(new_a, U_sub, V_sub)
        B = canonicalize(new_a)
        split_prob[B] = split_prob.get(B, 0.0) + (w / total_trees) / s
    return split_prob


def dual_graph(a, U, V):
    G = nx.Graph()
    G.add_nodes_from(np.unique(a))
    G.add_edges_from(district_adjacency(a, U, V))
    return G


def atlas_class(G, atlas):
    for j, H in enumerate(atlas):
        if nx.is_isomorphic(G, H):
            return j
    return None


def graph_thumbnail(G, px=50, node_size=18):
    fig, ax = plt.subplots(figsize=(px/100, px/100), dpi=100)
    nx.draw(G, pos=nx.kamada_kawai_layout(G), ax=ax,
            node_size=node_size, node_color='steelblue',
            edge_color='black', width=1.2)
    ax.set_axis_off()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', transparent=True, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()


def proportions(df, field, weight_col=None):
    if weight_col is None:
        s = df[field].value_counts(normalize=True)
    else:
        s = df.groupby(field)[weight_col].sum()
        s = s / s.sum()
    return s.rename('proportion').rename_axis(field).reset_index()

In [ ]:
filename = 'enum_Any[4,4]_4_rc.txt'
# filename = 'enum_Any[5,5]_5_rc.txt'

In [ ]:
assignments = load_grid(filename)
m, n = map(int, re.search(r'\[(\d+),(\d+)\]', filename).groups())
grid_tag = f'{m}x{n}'
plan_index = build_plan_index(assignments, m, n)
U, V = build_edges(m, n)
pi = ground_truth_distribution(assignments, m, n)

In [ ]:
N = len(assignments)
P = np.zeros((N, N))

k = len(np.unique(parse(assignments[0])))
target_size = (m * n) // k
D_total = k * (k - 1) // 2
shape_cache = {}

for idx, line in enumerate(assignments):
    a = parse(line)
    diag = 0.0
    for i, j in combinations(np.unique(a), 2):
        Wi = district_nodes(a, i)
        Wj = district_nodes(a, j)
        W = np.union1d(Wi, Wj)
        U_sub, V_sub = induced_edges(U, V, W)

        dist = split_distribution(a, W, U_sub, V_sub, i, j, target_size, shape_cache)
        diag += (1.0 - sum(dist.values())) / D_total
        for B, p in dist.items():
            P[idx, plan_index[B]] += p / D_total
    P[idx, idx] += diag

P_jump = P.copy()
np.fill_diagonal(P_jump, 0)
P_esc = P_jump / np.sum(P_jump, axis=1, keepdims=True)

In [ ]:
atlas = [G for G in graph_atlas_g() if G.number_of_nodes() == k]
graphs = [dual_graph(parse(line), U, V) for line in assignments]
class_id = np.array([atlas_class(G, atlas) for G in graphs])

dual_graph_df = pd.DataFrame({'iso_match': class_id, 'tree_weight': pi})

In [ ]:
tick_vals = sorted(dual_graph_df['iso_match'].unique())
tick_img_df = pd.DataFrame({
    'iso_match': tick_vals,
    'thumb': [graph_thumbnail(atlas[v]) for v in tick_vals]
})

image_axis = alt.Chart(tick_img_df).mark_image(width=42, height=42).encode(
    x=alt.X('iso_match:O', axis=None, sort=tick_vals),
    url='thumb:N'
).properties(width=400, height=55)

long_iso = pd.concat([
    proportions(dual_graph_df, 'iso_match').assign(weighting='uniform'),
    proportions(dual_graph_df, 'iso_match', 'tree_weight').assign(weighting='tree-weighted'),
], ignore_index=True)

edge_count_map = {v: atlas[v].number_of_edges() for v in tick_vals}
long_iso['edge_count'] = long_iso['iso_match'].map(edge_count_map)

edge_counts_sorted = sorted(set(edge_count_map.values()))
edge_color_scale = alt.Scale(
    domain=edge_counts_sorted,
    range=['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4'][:len(edge_counts_sorted)]
)

dual_graph_dist_w = alt.Chart(long_iso).mark_bar().encode(
    x=alt.X('iso_match:O', sort=tick_vals,
            axis=alt.Axis(labels=False, ticks=False, title=None)),
    y=alt.Y('proportion:Q'),
    color=alt.Color('edge_count:O', scale=edge_color_scale, legend=alt.Legend(title='Edges')),
    row=alt.Row('weighting:N', sort=['uniform', 'tree-weighted'], title=None)
).properties(width=400, height=150)

dual_graph_dist_w_imgticks = alt.vconcat(
    dual_graph_dist_w,
    image_axis,
    spacing=2
).resolve_scale(x='shared')

dual_graph_dist_w_imgticks.save(f'{grid_tag}dualgraphdistribution.png', ppi=300)
dual_graph_dist_w_imgticks

In [ ]:
classes = sorted(set(class_id.tolist()))
K = len(classes)
class_index = {c: i for i, c in enumerate(classes)}
class_pos = np.array([class_index[c] for c in class_id])

membership = np.zeros((N, K))
membership[np.arange(N), class_pos] = 1
pi_mass = membership.T @ pi
weights = membership * (pi / pi_mass[class_pos])[:, None]
P_block = weights.T @ P_esc @ membership

block_df = pd.DataFrame([
    {'from_class': fc, 'to_class': tc, 'probability': float(P_block[fi, ti])}
    for fi, fc in enumerate(classes)
    for ti, tc in enumerate(classes)
])
block_df['probability_plot'] = np.where(np.abs(block_df['probability']) <= 1e-12, 0.0, block_df['probability'])
block_df['xi'] = block_df['to_class'].map(class_index)
block_df['yi'] = block_df['from_class'].map(class_index)

thumb_df = pd.DataFrame({'class_id': classes, 'thumb': [graph_thumbnail(atlas[c]) for c in classes]})
thumb_df['i'] = thumb_df['class_id'].map(class_index)

cell_px = 46
img_px = 34
xs = alt.Scale(domain=[-1, K], nice=False, zero=False)
ys = alt.Scale(domain=[K, -1], nice=False, zero=False)
positive = block_df.loc[block_df['probability_plot'] > 0, 'probability_plot']
hidden_axis = alt.Axis(labels=False, ticks=False, grid=False, domain=False, title='')

heat = alt.Chart(block_df).mark_rect(width=cell_px, height=cell_px, stroke='white', strokeWidth=1.5).encode(
    x=alt.X('xi:Q', scale=xs, axis=hidden_axis),
    y=alt.Y('yi:Q', scale=ys, axis=hidden_axis),
    color=alt.condition(
        'datum.probability_plot === 0',
        alt.value('#ffffff'),
        alt.Color('probability_plot:Q',
                  scale=alt.Scale(type='linear', domain=[float(positive.min()), float(positive.max())],
                                  range=['#c6dbef', '#08306b'], clamp=True),
                  legend=alt.Legend(title='Transition probability', format='.3f', gradientLength=200)),
    ),
)

y_imgs = alt.Chart(thumb_df).mark_image(width=img_px, height=img_px).encode(
    x=alt.datum(-1), y=alt.Y('i:Q', scale=ys, axis=None), url='thumb:N'
)
x_imgs = alt.Chart(thumb_df).mark_image(width=img_px, height=img_px).encode(
    x=alt.X('i:Q', scale=xs, axis=None), y=alt.datum(K), url='thumb:N'
)

class_heatmap = alt.layer(heat, y_imgs, x_imgs).resolve_scale(color='independent').properties(
    width=cell_px * (K + 1),
    height=cell_px * (K + 1),
    title=f'District dual graph transitions, {grid_tag}_k{k}',
).configure_view(stroke=None)

class_heatmap.save(f'{grid_tag}_dualgraph_transition_no_self_loop.png', ppi=300)
class_heatmap